In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import gc
# import torch

# def clear_cuda_memory():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.synchronize()
#         torch.cuda.empty_cache()
#         torch.cuda.ipc_collect()

# clear_cuda_memory()

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import flexiznam as flz
from cottage_analysis.analysis import (
    spheres,
    find_depth_neurons,
)
from typing import Callable
from cottage_analysis.pipelines import pipeline_utils
from dataclasses import dataclass
from typing import Optional

In [ ]:
from matplotlib import pyplot as plt
import matplotlib as mpl  

In [ ]:
session_name="BRAC9972.2a_S20241021"
project="depth_mismatch_seq"
protocol_base="SpheresPermTubeReward"
photodiode_protocol=5
conflicts="skip"
filter_datasets = {"anatomical_only": 3, "ast_neuropil": False}
exclude_datasets = None

flexilims_session = flz.get_flexilims_session(project_id=project)

_, trials_df_all = spheres.sync_all_recordings(
    session_name=session_name,
    flexilims_session=flexilims_session,
    project=project,
    filter_datasets=filter_datasets,
    exclude_datasets=exclude_datasets,
    conflicts=conflicts,
    recording_type="two_photon",
    protocol_base=protocol_base,
    photodiode_protocol=photodiode_protocol,
    return_volumes=True,
)

In [ ]:
try:
    import torch
except Exception as e:
    raise ImportError(
        "This module requires PyTorch. Install with: pip install torch"
    ) from e



In [ ]:
from tqdm import tqdm
from cottage_analysis.analysis import torch_utils
from cottage_analysis.analysis import torch_fit_gaussian_blob as tfit

from cottage_analysis.analysis.torch_utils import Gaussian2DCholeskyBounds, Gaussian2DBounds, Gaussian1DBounds, GaussianAdditiveBounds, Gaussian2DAngleBounds

# Examine the goodness of fit on AdamW

First order estimations are great for complex models with many parameters but are not so good at curve fitting, which is what fit_gaussian_blob does. In my curve fitting approach, I want to 1) accelerate calculations with the GPU 2) be able to make several random initial guesses for fitting neural responses to each type of curve (model) and 3) implement some second order estimations of the curvature of the residual surface across the parameter space to prevent the fit getting stuck in local minima. Before resorting to this more computationally intensive approach, I want to exhaust faster methods that work out of the box with PyTorch. 

In [ ]:
rs_array, of_array, _, response_array, depth_list = tfit.process_rs_of_for_fit(trials_df_all)

In [ ]:
# make sure the data don't have any invalid running speed or optic flow values
if rs_array.ndim != 1 or of_array.ndim != 1 or depth_list.ndim != 1:
    raise ValueError("Stimulus arrays must be 1D.")
if response_array.ndim != 2:
    raise ValueError("Response array must be 2D with shape (n_samples, n_rois).")
if rs_array.shape[0] != of_array.shape[0] or rs_array.shape[0] != response_array.shape[0] or rs_array.shape[0] != depth_list.shape[0]:
    raise ValueError("Stimulus, response, and depth arrays must have the same number of samples.")
print("Validating stimulus arrays...", flush=True)
valid = (np.isfinite(rs_array) & np.isfinite(of_array))
if not np.any(valid):
    raise ValueError("No valid samples after filtering for finite values.")

rs_array = rs_array[valid]
of_array = of_array[valid]
response_array = response_array[valid, :]
depth_list = depth_list[valid]

# set up the boundary conditions based on the experimental conditions
PARAM_RANGE = {"rs_min": 0.005, "rs_max": 5, "of_min": 0.03, "of_max": 3000, "log_amplitude_max": 10.}

bounds = torch_utils.format_model_bounds("g2d", **PARAM_RANGE)
lower, upper = torch_utils.vectorise_bounds(bounds)
print(bounds, flush=True)

# set up the torch device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}", flush=True)
dtype = "float64"

torch_device = torch.device(device)
torch_dtype = torch.float64 if dtype == "float64" else torch.float32

# create tensors for stimuli and responses
X = (torch.as_tensor(rs_array, device=torch_device, dtype=torch_dtype), torch.as_tensor(of_array, device=torch_device, dtype=torch_dtype))
y = torch.as_tensor(response_array, device=torch_device, dtype=torch_dtype)

n_samples, n_rois = y.shape
print(f"n_samples={n_samples}, n_rois={n_rois}", flush=True)

In [ ]:
# Load the neurons_df to choose neurons that are actually depth selective and therefore should have fits that converge
neurons_df_path = "/nemo/lab/znamenskiyp/home/shared/projects/depth_mismatch_seq/BRAC9972.2a/S20241021/neurons_df.pickle"
neurons_df = pd.read_pickle(neurons_df_path)

# neurons_df[neurons_df["iscell"] == 1].sort_values(by="rsof_rsq_closedloop_g2d", ascending=False).head()

NICE_ROIS = [495, 519, 853, 1285, 155]

print(f"Production code R^2: {np.nanmedian(neurons_df.rsof_rsq_closedloop_crossval_g2d)}")

In [ ]:
# take the top 100 well-fit ROIs, we expect these fits to converge regardless of whether we use scipy or torch
# rois = neurons_df[neurons_df["iscell"] == 1].sort_values("rsof_rsq_closedloop_g2d", ascending=False).head(100).roi.values
# N_ROIS_SUB = 100
# ROI_SUBSET = torch.tensor(rois)

# print(f"Production code R^2 of top {N_ROIS_SUB} well-fit ROIs: {np.median(neurons_df.loc[neurons_df['roi'].isin(rois), 'rsof_rsq_closedloop_g2d'].values)}")


In [ ]:
import cottage_analysis.analysis.torch_utils as torch_utils

In [ ]:
N_STARTS=5
init_params = torch_utils.generate_n_inits_all_rois(
    n_starts=N_STARTS,
    X=X,
    y=y[:, NICE_ROIS],
    model="g2d",
    bounds=bounds,
    rng_seed=42,
    device=torch_device,
    dtype=torch_dtype,
    apply_bounds=True,
)
init_params = torch_utils.decode_params(init_params, model="g2d", bounds=bounds)

In [ ]:
from cottage_analysis.analysis.torch_fit_gaussian_blob import gaussian_2d_cholesky

model_func = gaussian_2d_cholesky

In [ ]:
# Do TRF curve fitting on a good ROI only with N_STARTS
trf_fit = torch_utils.Curve_fit(
    X=X,
    y=y[:, NICE_ROIS],
    params=init_params,
    bounds=bounds,
    model_func=model_func,
    n_starts=N_STARTS,
    n_iters=1000,
    method="trf",
)

params_fit = trf_fit.fit()
r2_final = trf_fit.r2.view(len(NICE_ROIS), N_STARTS)
best_r2, best_idx = r2_final.max(dim=1)
best_params = params_fit.view(len(NICE_ROIS), N_STARTS, -1)[torch.arange(len(NICE_ROIS)), best_idx]

print("Best R2:", best_r2)
print("Best Parameters:", best_params)

In [ ]:
print(f"Production code R^2: {neurons_df.rsof_rsq_closedloop_g2d.apply(lambda x: np.nanmedian(x)).values[NICE_ROIS]}")

# Check that there is an actual performance gap

Choose a single ROI and use the same restart to fit the g2d model using production code in `fit_gaussian_blob` with `scipy` and the `torch` model. Compare the fit parameters and the R-squared. 

In [ ]:
import functools
from cottage_analysis.analysis.fit_gaussian_blob import initial_fit_conditions
from cottage_analysis.analysis.fit_gaussian_blob import gaussian_2d as gaussian_2d_scipy
from scipy.optimize import curve_fit
from cottage_analysis.analysis import spheres, common_utils

In [ ]:
# build the shared starting point
dff_roi = response_array[:, 155]
MIN_SIGMA = 0.25
_, lower_bounds, upper_bounds, p0_func = initial_fit_conditions(
    "gaussian_2d", param_range=PARAM_RANGE
)
p0 = p0_func(X=(rs_array, of_array), y=dff_roi, i_iter=0)
p0_arr = np.array(p0, dtype=np.float64)
lower_arr = np.array(lower_bounds, dtype=np.float64)
upper_arr = np.array(upper_bounds, dtype=np.float64)

print("Shared starting point p0:")
for name, val in zip(p0._fields, p0_arr):
    print(f"  {name:>14s}: {val: .4f}")

In [ ]:
scipy_model = functools.partial(gaussian_2d_scipy, min_sigma=MIN_SIGMA)
popt_scipy, _ = curve_fit(
    scipy_model,
    (rs_array, of_array),
    dff_roi,
    p0=p0_arr,
    bounds=(lower_arr, upper_arr),
    maxfev=3000,
)
pred_scipy = scipy_model((rs_array, of_array), *popt_scipy)
r2_scipy = common_utils.calculate_r_squared(dff_roi, pred_scipy)

print("scipy popt:", popt_scipy)
print(f"scipy R^2 from shared p0: {r2_scipy:.4f}")

In [ ]:
model_func = tfit.gaussian_2d_cholesky

params_shared = torch.tensor(p0_arr, device=torch_device, dtype=torch_dtype).unsqueeze(0)
y_roi_torch = y[:, 155:156]

trf_fit_shared = torch_utils.Curve_fit(
    X=X,
    y=y_roi_torch,
    params=params_shared,
    bounds=bounds,
    model_func=model_func,
    n_starts=1,
    n_iters=1000,
    method="trf",
)
params_fit_shared = trf_fit_shared.fit()
r2_torch_shared = trf_fit_shared.r2

print("torch popt:", params_fit_shared.detach().cpu().numpy().ravel())
print(f"torch R^2 from shared p0: {r2_torch_shared.item():.4f}")

In [ ]:
param_names = ["log_amplitude", "x0", "y0", "log_l11", "l21", "log_l22", "offset"]
popt_torch = params_fit_shared.detach().cpu().numpy().ravel()

print(f"{'param':>14s}  {'scipy':>12s}  {'torch':>12s}  {'diff':>12s}")
for name, v_s, v_t in zip(param_names, popt_scipy, popt_torch):
    print(f"{name:>14s}  {v_s: 12.4f}  {v_t: 12.4f}  {v_t - v_s: 12.4f}")

print(
    f"\nR^2  -- scipy: {r2_scipy:.4f}   torch: {r2_torch_shared.item():.4f}   "
    f"diff: {r2_torch_shared.item() - r2_scipy:.4f}"
)

if abs(r2_torch_shared.item() - r2_scipy) < 0.01:
    print(
        "\n=> Converged to the same point/R^2 from the same start."
    )
else:
    print(
        "\n=> Torch diverged meaningfully from the same start."
    )

# Compare the gaussian blob shape

`log_l11`, `l21`, and `log_l22` are the parameters controlling the shape of the Gaussian blob but can disagree wildly from fit to fit, while the R-sqaured
is identical to 4 decimal places. To compare the shape of the Gaussian blob between the two fits, we can reconstruct the covariance matrix from the fit params and then get the angle
parameterised fit values for comparison.

In [ ]:
from cottage_analysis.analysis.fit_gaussian_blob import (
    _effective_precision_eig,
    get_gaussian_angle,
    get_semimajor_length,
    get_semiminor_length,
)

print(f"Saturation eigenvalue (1/(2*min_sigma)) = {1.0 / (2 * MIN_SIGMA):.4f}\n")

results = {}
for label, popt in [("scipy", popt_scipy), ("torch", popt_torch)]:
    eigvals, _ = _effective_precision_eig(popt, min_sigma=MIN_SIGMA)
    angle = get_gaussian_angle(popt, min_sigma=MIN_SIGMA)
    semimajor = get_semimajor_length(popt, min_sigma=MIN_SIGMA)
    semiminor = get_semiminor_length(popt, min_sigma=MIN_SIGMA)
    results[label] = (eigvals, angle, semimajor, semiminor)
    print(
        f"{label:>6s}: effective eigenvalues={eigvals}, angle={angle:.2f} deg, "
        f"semimajor={semimajor:.4f}, semiminor={semiminor:.4f}"
    )

eig_scipy, angle_scipy = results["scipy"][0], results["scipy"][1]
eig_torch, angle_torch = results["torch"][0], results["torch"][1]
eig_diff = np.abs(eig_scipy - eig_torch).max()
angle_diff = abs(angle_scipy - angle_torch)

print(f"\nMax abs diff in effective eigenvalues: {eig_diff:.4f}")
print(f"Angle diff: {angle_diff:.2f} deg")

if eig_diff < 0.05 and angle_diff < 5:
    print("Effective shape is basically the same.")
else:
    print("Effective shape differs significantly.")